# Сборка пар pRESTO — человек (`PRJEB30386`)

`AssemblePairs.py align` получает синхронизированные `pr_trimmed` парные риды с SRA-координатой и pRESTO-аннотациями. Используются `--coord sra`, `--rc tail` и перенос `BARCODE` из R2; каждые 30 секунд печатается сигнал активности. Перед публикацией проверяются код завершения, маркер `END> AssemblePairs`, gzip/FASTQ и полный `assembly_qc.tsv`.

In [ ]:
import os, sys, sysconfig, shutil, subprocess, time
from pathlib import Path

_ENV_CANDIDATES = ["/opt/conda/envs/bcr_env", "/Users/epishkin/mamba/envs/bcr_env"]
BCR_ENV = next((Path(p) for p in _ENV_CANDIDATES if (Path(p) / "bin").is_dir()), None)
if BCR_ENV is None:
    raise FileNotFoundError(f"Среда bcr_env не найдена: {_ENV_CANDIDATES}")
os.environ["PATH"] = str(BCR_ENV / "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
for _site in sorted((BCR_ENV / "lib").glob("python*/site-packages")):
    if str(_site) not in sys.path:
        sys.path.insert(0, str(_site))
print("BCR_ENV:", BCR_ENV)
DATASET = "PRJEB30386"
LOCAL_REPO = Path("/Users/epishkin/workspace/bcr-assembler")
VOLUME_ROOT = Path("/data/user/epishkin")

def _has_fastq(path, pattern="*.fastq.gz"):
    return Path(path).is_dir() and any(Path(path).glob(pattern))

def resolve_raw_and_result():
    remote_raw = VOLUME_ROOT / "raw" / DATASET
    local_result = LOCAL_REPO / "results" / DATASET
    local_raw = LOCAL_REPO / "raw" / DATASET
    if _has_fastq(remote_raw, "*_1.fastq.gz"):
        return remote_raw, VOLUME_ROOT / "results" / DATASET
    if _has_fastq(local_raw, "*_1.fastq.gz"):
        return local_raw, local_result
    raise FileNotFoundError(f"Парные FASTQ не найдены в {remote_raw} или {local_raw}")

def resolve_result():
    remote = VOLUME_ROOT / "results" / DATASET
    local = LOCAL_REPO / "results" / DATASET
    return remote if remote.is_dir() else local


In [ ]:
import csv, gzip, re

NPROC = 8
HEARTBEAT = 30
WRITE_FAILED_READS = True
FORCE = False
print("AssemblePairs.py:", shutil.which("AssemblePairs.py"))


In [ ]:
def _tool(name):
    path = shutil.which(name)
    if not path:
        raise FileNotFoundError(f"Не найден executable: {name} (BCR_ENV={BCR_ENV})")
    return path

def _run_visible(cmd, stdout_log, stderr_log, outputs=(), heartbeat=30):
    stdout_log, stderr_log = Path(stdout_log), Path(stderr_log)
    stdout_log.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    print("[run]", " ".join(map(str, cmd)), flush=True)
    with stdout_log.open("w") as stdout, stderr_log.open("w") as stderr:
        proc = subprocess.Popen([str(x) for x in cmd], stdout=stdout, stderr=stderr, text=True)
        print(f"PID={proc.pid}", flush=True)
        while proc.poll() is None:
            sizes = " ".join(
                f"{Path(x).name}={Path(x).stat().st_size / 1e6:.1f}MB"
                for x in outputs if Path(x).exists()
            )
            print(f"PID={proc.pid} elapsed={(time.monotonic()-started)/60:.1f}min {sizes}", flush=True)
            time.sleep(heartbeat)
    if proc.returncode:
        raise RuntimeError(f"rc={proc.returncode}; см. {stderr_log}")

def _promote(staging, final):
    staging, final = Path(staging), Path(final)
    previous = final.parent / f".{final.name}.previous"
    if previous.exists():
        shutil.rmtree(previous)
    if final.exists():
        final.rename(previous)
    try:
        staging.rename(final)
    except Exception:
        if previous.exists() and not final.exists():
            previous.rename(final)
        raise
    if previous.exists():
        shutil.rmtree(previous)

def _count_fastq(path):
    path = Path(path)
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as handle:
        lines = sum(1 for _ in handle)
    if lines % 4:
        raise RuntimeError(f"Повреждён FASTQ: {path}")
    return lines // 4

def _single(pattern):
    hits = sorted(pattern.parent.glob(pattern.name))
    if len(hits) != 1:
        raise RuntimeError(f"Ожидался один файл {pattern}, найдено: {hits}")
    return hits[0]

def _terminal_metric(text, key):
    hits = re.findall(rf"(?m)^\s*{key}>\s*([0-9,]+)\s*$", text)
    if not hits:
        raise RuntimeError(f"В stdout отсутствует итоговая метрика {key}>")
    return int(hits[-1].replace(",", ""))

def run_presto_merge(force=FORCE):
    result = resolve_result()
    src = result / "pr_trimmed" / "fastq"
    final, staging = result / "merged", result / ".merged.staging"
    samples = sorted(p.name.removesuffix("_1.pr.fastq.gz") for p in src.glob("*_1.pr.fastq.gz"))
    if not samples or any(not (src / f"{s}_2.pr.fastq.gz").is_file() for s in samples):
        raise RuntimeError(f"Неполный набор синхронизированных пар: {src}")
    if staging.exists():
        if not force:
            raise FileExistsError(f"Остался staging: {staging}; установите FORCE=True для очистки")
        shutil.rmtree(staging)
    out, logs, qc = staging / "fastq", staging / "logs", staging / "qc"
    for d in (out, logs, qc):
        d.mkdir(parents=True, exist_ok=True)
    rows = []
    for sample in samples:
        r1, r2 = src / f"{sample}_1.pr.fastq.gz", src / f"{sample}_2.pr.fastq.gz"
        in1, in2 = _count_fastq(r1), _count_fastq(r2)
        if in1 == 0 or in1 != in2:
            raise RuntimeError(f"Вход {sample} не синхронизирован: R1={in1}, R2={in2}")
        stdout_log, stderr_log = logs / f"{sample}.assemble.stdout.log", logs / f"{sample}.assemble.stderr.log"
        cmd = [_tool("AssemblePairs.py"), "align", "-1", r1, "-2", r2,
               "--coord", "sra", "--rc", "tail", "--2f", "BARCODE", "--outname", sample,
               "--outdir", out, "--nproc", str(NPROC), "--gzip-output"]
        if WRITE_FAILED_READS:
            cmd.append("--failed")
        _run_visible(cmd, stdout_log, stderr_log, tuple(out.glob(f"{sample}*")), heartbeat=HEARTBEAT)
        stdout_text = stdout_log.read_text(errors="replace")
        if "END> AssemblePairs" not in stdout_text:
            raise RuntimeError(f"Нет маркера завершения для {sample}: {stdout_log}")
        terminal_pairs = _terminal_metric(stdout_text, "PAIRS")
        terminal_pass = _terminal_metric(stdout_text, "PASS")
        terminal_fail = _terminal_metric(stdout_text, "FAIL")
        if terminal_pairs != in1 or terminal_pass + terminal_fail != terminal_pairs:
            raise RuntimeError(
                f"Итоги {sample} несогласованы: input={in1}, PAIRS={terminal_pairs}, "
                f"PASS={terminal_pass}, FAIL={terminal_fail}"
            )
        pass_fastq = _single(out / f"{sample}*assemble-pass.fastq.gz")
        assembled = _count_fastq(pass_fastq)
        if assembled != terminal_pass or not pass_fastq.stat().st_size:
            raise RuntimeError(f"PASS {sample}: stdout={terminal_pass}, FASTQ={assembled}")
        fail_files = sorted(out.glob(f"{sample}*assemble-fail.fastq.gz"))
        if WRITE_FAILED_READS:
            if len(fail_files) != 2:
                raise RuntimeError(f"Ожидались два mate fail FASTQ для {sample}: {fail_files}")
            fail_counts = [_count_fastq(path) for path in fail_files]
            if fail_counts != [terminal_fail, terminal_fail]:
                raise RuntimeError(f"FAIL {sample}: stdout={terminal_fail}, mate FASTQ={fail_counts}")
        rows.append({"sample": sample, "input_pairs": terminal_pairs,
                     "assembled_pairs": terminal_pass, "failed_pairs": terminal_fail,
                     "merge_rate": f"{terminal_pass / terminal_pairs:.6f}",
                     "pass_fastq": pass_fastq.name,
                     "completion_marker": "END> AssemblePairs"})
        qc_path = qc / "assembly_qc.tsv"
        with qc_path.open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]), delimiter="\t")
            writer.writeheader(); writer.writerows(rows)
    if len(rows) != len(samples) or len(list(out.glob("*_assemble-pass.fastq.gz"))) != len(samples):
        raise RuntimeError("Проверка полноты merged не пройдена")
    qc_path = qc / "assembly_qc.tsv"
    with qc_path.open() as handle:
        if sum(1 for _ in handle) != len(samples) + 1:
            raise RuntimeError("assembly_qc.tsv неполон")
    _promote(staging, final)
    print("Готово:", final)


## Запуск

Существующая `merged` не удаляется до полной проверки новой staging-стадии.

In [ ]:
run_presto_merge(force=FORCE)
